In [ ]:
# --- path configuration -------------------------------------------------
# Every path goes through resolve() (inputs under data/) or out_path() (this run's output).
from surfaceome_config import resolve, out_path


In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

In [ ]:
# generated tables are written through out_path(); see the interaction_heights cells below


In [ ]:
#STRING v12 links between surfaceome proteins, each pair labelled from the decision tables
#(build_string_interactions.py): a pair with no call has an empty label and never counts as trans
human_interactions = pd.read_csv(resolve('inputs/string/string_surfaceome_pairs.csv'), keep_default_na = False,
                                 na_values = {'combined_score_human': ['']})
#the 2022 baseline's table carries 46 rows added by hand in 2020 ('annotated'), superseded by curated_trans_interactions.csv
human_interactions = human_interactions[human_interactions['_merge'] != 'annotated']

#pairs the 2020 labels got wrong, relabelled with the evidence; see data/README.md
from surfaceomeTopography.interactions import correct_interaction_types
human_interactions = correct_interaction_types(human_interactions,
    pd.read_csv(resolve('curated/interaction_type_corrections.csv')))

#the height table keyed by accession, with each protein's AlphaFold model id -- the same view the
#CellphoneDB mapping notebook sizes its interactions from, built by one helper for both
from surfaceomeTopography.interactions import estimates_with_af_ids
heights = estimates_with_af_ids(
    pd.read_csv(str(resolve('database/height_estimates.csv'))),
    pd.read_csv(resolve('inputs/proteome/surfaceome_current.tsv'), sep = '\t', header = None, names = ['current', 'af_id']))


### Modify integrin heights to reflect the extended confirmation

## Selection and filtering of trans interactions

In [ ]:
#high confidence interactions only
high_conf = human_interactions[human_interactions['combined_score_human'] >= 400]

#high confidence trans interactions
trans = high_conf[(high_conf['Interaction Type (cis, trans, or secreted)'] == 'trans')]

#add antibody bridges: each cited antigen of a therapeutic antibody with each cited Fc-gamma receptor
from surfaceomeTopography.interactions import antibody_bridge_pairs
fcr = antibody_bridge_pairs(pd.read_csv(resolve('curated/antibody_bridges.csv')))

#add the pairs curated by hand, each with its citations; a pair recorded with included = no was checked and is left out
#(see data/README.md, "Custom rules", and data/CITATIONS.md)
curated = pd.read_csv(resolve('curated/curated_trans_interactions.csv'))
add = curated[curated['included'] == 'yes']

trans = pd.concat([fcr,add, trans])

trans.loc[:,'Human interaction_id'] = trans.apply(lambda row: ','.join(sorted([row['Human ID link_prot1'], row['Human ID link_prot2']])), axis = 1)
trans.head()                                                                                  

In [ ]:
trans_h1 = pd.merge(trans[['Human ID link_prot1', 'Human gene name_prot1',
                          'Human ID link_prot2', 'Human gene name_prot2',
                          'Interaction Type (cis, trans, or secreted)', 'Entry name_prot1',
                          'Human interaction_id', '_merge']], 
                    heights[['ID link', 'Entry name_first',
                           'total_height', 
                             'ID']],
                    left_on = 'Human ID link_prot1',
                    right_on = 'ID link',
                    how = 'inner')

trans_h2 = pd.merge(trans[['Human ID link_prot1', 'Human gene name_prot1',
                          'Human ID link_prot2', 'Human gene name_prot2',
                          'Interaction Type (cis, trans, or secreted)', 'Entry name_prot2',
                          'Human interaction_id', '_merge']], 
                    heights[['ID link', 'Entry name_first',
                            'total_height', 
                             'ID', ]],
                    left_on = 'Human ID link_prot2',
                    right_on = 'ID link',
                    how = 'inner')

trans_heights = pd.merge(trans_h1, trans_h2, suffixes = ('_prot1', '_prot2'), on = ['Human ID link_prot1',
                                                                                    'Human gene name_prot1',
                                                                                    'Human ID link_prot2', 
                                                                                    'Human gene name_prot2',
                                                                                    'Interaction Type (cis, trans, or secreted)',
                                                                                    'Human interaction_id', '_merge'
                                                                                   ])


#calculate height by addition
trans_heights['interaction_dim'] = trans_heights['total_height_prot1'] + trans_heights['total_height_prot2']

trans_heights.drop_duplicates(subset=['Entry name_first_prot1', 'Entry name_first_prot2', 'Human ID link_prot1', 'Human ID link_prot2'], inplace = True)

#add antibody aditional height (3.24 nm based on 5DK3, 4X4M)
trans_heights = trans_heights.assign(interaction_dim = np.where(trans_heights._merge == 'FcR', trans_heights['interaction_dim']+ 3.24, trans_heights['interaction_dim']))

trans_heights.drop_duplicates('Human interaction_id', inplace = True)

trans_heights.rename(columns = {'Entry name_first_prot2': 'Entry_prot2',
                               'Entry name_first_prot1': 'Entry_prot1',
                               }, inplace = True)


#trans_heights = trans_heights.assign(Entry_prot1 = np.where(trans_heights['Entry_prot1'].isnull(), trans_heights['Entry name_prot1'], trans_heights['Entry_prot1']))


trans_heights.head()

In [ ]:
#add cellphoneDB trans interactions

##need to update these with the updated heights...done 
#check on cellphoneDB updates?...done, no updates?
cellphoneDB = pd.read_csv(str(resolve('database/cellphonedb_tm_interaction_sizes.csv')))

cellphoneDB.loc[:,'Interaction Type (cis, trans, or secreted)'] = 'trans'
cellphoneDB.rename(columns = {"ID_link_prot1": 'Human ID link_prot1',
                             "ID_link_prot2": 'Human ID link_prot2',
                             "intact_height": "interaction_dim",
                             'total_height_a': 'total_height_prot1',
                              'total_height_b': 'total_height_prot2',
                             },
                  inplace = True)


cellphoneDB.head()

In [ ]:
#one table of trans interactions: a pair in both STRING and CellphoneDB is one row, matched on its two
#accessions in either order, not on its computed heights
from surfaceomeTopography.interactions import combine_interaction_sources, drop_corrected_pairs
#a pair the corrections table says is not trans (CD80-PD-L1 binds only in cis) leaves the CellphoneDB rows too
cellphoneDB = drop_corrected_pairs(cellphoneDB,
    pd.read_csv(resolve('curated/interaction_type_corrections.csv')))
comb = combine_interaction_sources(trans_heights, cellphoneDB[['id_cp_interaction', 'partner_a', 'partner_b', 'annotation_strategy', 'source',
       'total_height_prot1', 'total_height_prot2',
       'interaction_dim', 'Human ID link_prot1',
       'Human ID link_prot2', 'Human interaction_id', 'Entry_prot1',
       'Entry_prot2',
       'Interaction Type (cis, trans, or secreted)']])

In [ ]:
#interaction heights set other than by the sum of the two partners, each rule recorded with its evidence
#in interaction_height_rules.csv; see data/README.md, "Custom rules"
from surfaceomeTopography.interactions import apply_height_rules
height_rules = pd.read_csv(resolve('curated/interaction_height_rules.csv'))
comb = apply_height_rules(comb, height_rules)

height_rules[['rule', 'gene', 'partner_genes', 'pmids']]

In [ ]:
trans_heights.to_csv(out_path('database/interaction_heights_trans_all.csv'), index = False)

In [ ]:
comb.to_csv(out_path('database/interaction_heights.csv'), index = False)

In [ ]:
#where each antibody's epitope stands on its antigen: the residues of curated/antibody_epitopes.csv, read
#part-way up the antigen's stack (surfaceomeTopography.epitope); a bridge's gap is set from it at a contact
from surfaceomeTopography.epitope import curated_epitope_heights
epitope_heights = curated_epitope_heights(
    pd.read_csv(resolve('curated/antibody_epitopes.csv'), dtype = str, keep_default_na = False),
    pd.read_csv(resolve('database/height_estimates.csv')),
    pd.read_csv(resolve('database/domain_disorder_assignments_alphafold.csv'), index_col = 0, low_memory = False),
    pd.read_csv(resolve('database/UP000005640_surfaceome_largest_ecds_classified.csv')),
    pd.read_csv(resolve('inputs/alphafold/alphafold_axis_positions.csv')))
epitope_heights.to_csv(out_path('database/antibody_epitope_heights.csv'), index = False)
epitope_heights

In [ ]:
import plotly.figure_factory as ff
import numpy as np

hist_data = comb.drop_duplicates('Human interaction_id')['interaction_dim'].dropna(),
            

group_labels = ['measured'] # name of the dataset

fig = ff.create_distplot(hist_data, 
                         group_labels, 
                         bin_size=[2]
                        )
fig.show()

In [ ]:
import plotly.figure_factory as ff
import numpy as np

hist_data = trans_heights.drop_duplicates('Human interaction_id')['interaction_dim']

group_labels = ['measured'] # name of the dataset

fig = ff.create_distplot([hist_data], 
                         group_labels, 
                         bin_size=[2]
                        )
fig.show()

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline


plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

hist_data = comb[~(comb['_merge'] == 'FcR')].drop_duplicates('Human interaction_id')['interaction_dim']

#histogram on linear scale

plt.figure(figsize=(15,15))

 
x = hist_data


#plt.subplot(211)
hist, bins, _ = plt.hist(x, bins=50, alpha = 0.5)
plt.plot(x, np.zeros(x.shape), 'b|', ms=20, color = '#00A8A8')
#plt.yscale('log')

# histogram on log scale. 
# Use non-equal bin sizes, such that they look equal on log scale.
logbins = np.logspace(np.log10(bins[0]),np.log10(bins[-1]),len(bins))
plt.subplot(212)
plt.hist(x, bins=logbins, histtype = 'step', color = '#59656F')



plt.xscale('log')
#plt.xlabel('height (nm)')
plt.legend(['interactions'])

plt.savefig(out_path('figures/03_heights_by_method_interactions.pdf'), transparent = True)

In [ ]:
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42


# histogram on linear scale
#plt.figure(figsize=(10,10))

fig = plt.figure(figsize=(10,8), constrained_layout=True)
gs = fig.add_gridspec(6, 1, figure = fig)

ax1 = fig.add_subplot(gs[0:1, 0])
ax1.plot(x, np.zeros(x.shape), 'b|', ms=20, color = '#59656F')  # rug plot
#ax1.set_ylim([-0.01, 0.01])
#hist, bins, _ = plt.hist(x, bins=50)
#plt.yscale('log')
plt.xlabel('height (nm)')
#plt.ylabel('log count')
#bins = range(1,1000,10)

# histogram on log scale. 
# Use non-equal bin sizes, such that they look equal on log scale.
logbins = np.logspace(np.log10(bins[0]),np.log10(bins[-1]),len(bins))


ax2 = fig.add_subplot(gs[1:4, 0])
plt.hist(x, bins=logbins, histtype = 'step', color = '#59656F')
plt.xscale('log')
plt.xlabel('log height (nm)')
plt.ylabel('count')
#plt.setp(ax2.get_xticklabels(), visible=False)



plt.savefig(out_path('figures/03_heights_interactions_breakdown.pdf'),
           transparent = True)

In [ ]:
td = comb.drop_duplicates('Human interaction_id').sort_values('interaction_dim').reset_index()
td['names'] = td['Entry_prot1'].str[:-6] +','+ td['Entry_prot2'].str[:-6]
td = td.set_index('names')

td.tail()

In [ ]:
import seaborn as sns
import scipy as sp

#sns.clustermap(trans_heights_well[['interaction_dim', 'Human interaction_id']].corr())

In [ ]:
h = heights.copy()
h['Entry name'] = heights['Entry name_first'].str[:-6]
p = h.sort_values('total_height').set_index('Entry name')
p.head()

In [ ]:
#compare trans heights to protein heights
df = pd.DataFrame()

for r in range(0,td['interaction_dim'].size):
    print(r)
    print(td.index.values[r])
    first_row = td['interaction_dim'][r]
    membrane_gap = td['interaction_dim'].apply(lambda row: row - first_row)
    df[td.index.values[r]] = membrane_gap
    
#df['names'] = td['Human interaction_id']
df

In [ ]:
p_vs = pd.DataFrame()

for r in range(0,p['total_height'].size):
    print(r)
    print(p.index.values[r])
    first_row = p['total_height'][r]
    membrane_gap = td['interaction_dim'].apply(lambda row: row - first_row)
    p_vs[p.index.values[r]] = membrane_gap
    
#df['names'] = td['Human interaction_id']
p_vs

In [ ]:
df = pd.DataFrame()


for r in range(0,td['interaction_dim'].size):
    print(r)
    print(td.index.values[r])
    first_row = td['interaction_dim'][r]
    membrane_gap = td['interaction_dim'].apply(lambda row: first_row - row)
    df[td.index.values[r]] = membrane_gap
    
#df['names'] = td['Human interaction_id']
df

In [ ]:
#data = df.apply()

# http://matplotlib.org/examples/pylab_examples/pcolor_log.html
# modified to use seaborn

import matplotlib.pyplot as plt
from matplotlib.colors import SymLogNorm
import numpy as np
import seaborn as sns; sns.set()

plt.figure(figsize=(20,20))

ax = sns.heatmap(df.iloc[:,:], center = 0, cmap = "RdBu_r",  
            norm=SymLogNorm(linthresh = 5, vmin=df.values.min(), vmax=df.values.max()),
            square = True
           #robust = True
           )

ax.invert_yaxis()

In [ ]:
#data = df.apply()

# http://matplotlib.org/examples/pylab_examples/pcolor_log.html
# modified to use seaborn

import matplotlib.pyplot as plt
from matplotlib.colors import SymLogNorm
import numpy as np
import seaborn as sns; sns.set()

plt.figure(figsize=(20,20))

ax = sns.clustermap(df.iloc[:,:], center = 0, cmap = "RdBu_r",  
            norm=SymLogNorm(linthresh = 5, vmin=df.values.min(), vmax=df.values.max()),
            square = True
           #robust = True
           )

#ax.invert_yaxis()

In [ ]:
plt.figure(figsize=(20,15))

ax = sns.heatmap(p_vs.iloc[:,:], 
                 #center = 0, 
                 cmap = "RdBu",  
            norm=SymLogNorm(linthresh = 3, vmin=-50, vmax= 50),
            #square = True
           #robust = True
                 
           )

ax.invert_yaxis()

In [ ]:
#the 2026 proteome, not the 2022 'output/' export: same columns, and this was the only read
#left on that stale copy.  Names only -- nothing this notebook writes depends on it.
proteome = pd.read_csv(str(resolve('inputs/proteome/UP000005640.tab')),
                      sep = '\t')


In [ ]:
heights_named = pd.merge(heights, proteome[['Entry', 'Gene names', 'Protein names']], left_on = 'ID link', right_on = 'Entry')
heights_named.head()

In [ ]:
surf = heights_named.sort_values('total_height')
surf[surf['Gene names'] == 'GPC2']

In [ ]:
surf = heights_named.sort_values('total_height')
surf[surf['Gene names'] == 'GPC2']

surf = surf.set_index('Protein names')

surf_vs = pd.DataFrame()

for r in range(0,surf['total_height'].size):
    print(r)
    print(surf.index.values[r])
    first_row = surf['total_height'][r]
    membrane_gap = td['interaction_dim'].apply(lambda row: row - first_row)
    surf_vs[surf.index.values[r]] = membrane_gap
    

surf_vs.head()

In [ ]:
plt.figure(figsize=(30,20))

ax = sns.heatmap(p_vs.iloc[:,:], 
                 #center = 0, 
                 cmap = "RdBu_r",  
            norm=SymLogNorm(linthresh = 1, vmin=p_vs.values.min(), vmax = p_vs.values.max()),
            #square = True
           #robust = True
           )

ax.invert_yaxis()

In [ ]:
plt.figure(figsize=(40,40))
ax = sns.clustermap(p_vs.iloc[:,:], 
                 #center = 0, 
                 cmap = "vlag_r",  
           norm=SymLogNorm(linthresh =0.5, vmin=p_vs.values.min(), vmax = p_vs.values.max()))

In [ ]:
import plotly.graph_objects as go

fig = go.Figure(data=go.Heatmap(
                z = p_vs.iloc[:,:],
                x = p_vs.columns,
                y = p_vs.index,
    
    colorscale = [[0.0, 'rgb(103,0,31)'], 
                  [0.29, 'rgb(178,24,43)'], 
                  [0.39, 'rgb(214,96,77)'], 
                  [0.49,'rgb(244,165,130)'], 
                  [0.499, 'rgb(253,219,199)'], 
                  
                  [0.5,'rgb(247,247,247)'], 
                  
                  [0.501,'rgb(209,229,240)'], 
                  [0.51,'rgb(146,197,222)'], 
                  [0.61,'rgb(67,147,195)'], 
                  [0.71,'rgb(33,102,172)'], 
                  [1,'rgb(5,48,97)']
                 ],
    
                zmid = 0,
                zmin = -268,
                zmax = 268

                   ))

import plotly.io as pio
# The interactive heatmap is kept as a file; auto_open=False so a run does not launch a browser.
pio.write_html(fig, file=str(out_path('figures/03_trans_vs_proteins_heatmap.html')), auto_open=False)               
#fig.show()